# Notebook 3: Đánh giá & Trực quan hóa đối chiếu 3 mô hình sinh ảnh `RGB (3ch) → IR (1ch)`: CycleGAN vs F-LSeSim vs StegoGAN

Notebook độc lập chuyên phục vụ **Kiểm định trực quan (mắt thường)** và **Đo lại toàn bộ Metrics định lượng** (cả 2 cách đánh giá) trên tập validation FLIR (`align_validation.txt`) cho các trọng số bạn đã train (kể cả trọng số lưu lại từ phiên bị timeout giữa chừng!).

### Những điểm tiện dụng đặc biệt của Notebook này:
1. **Nạp tự động MỌI loại file `.pth` bạn giữ được sau timeout:**
   - File chuẩn `best_net_G_A.pth` / `latest_net_G_A.pth` / `epoch_XXX_net_G_A.pth` (`CycleGANGenerator` 3ch $\to$ 1ch).
   - File resume **`flsesim_resume_latest.pth`** (tự rút `netG` và in rõ số epoch đã đạt!).
   - File resume **`stegogan_resume_latest.pth`** (tự dựng `StegoResnetMaskV1Generator`, ánh xạ nhánh sạch `fake_B_clean` sang `CycleGANGenerator` và in rõ số epoch đã đạt!).
   - Bạn thậm chí có thể khai báo đồng thời nhiều mốc (ví dụ cả `F-LSeSim (best)` và `F-LSeSim (latest)`) ngay trong `CONFIG["generators"]` để so sánh cùng lúc!
2. **Đánh giá bằng mắt thường cực trực quan (`Cell 5` & `Cell 6`):**
   - **Hình 1 (Toàn khung sạch — không vẽ box che mắt):** Đặt ngang hàng `RGB gốc` | `Real IR (1-ch thật)` | `CycleGAN` | `F-LSeSim` | `StegoGAN` trên đa dạng cảnh ngày/đêm, người/xe/xe đạp.
   - **Hình 2 (Zoom cận cảnh vật thể nhỏ `person` / `bicycle` / `car`):** Soi trực tiếp độ sắc nét khung xe đạp, dáng người đi bộ và khả năng đảo cực sáng–tối.
   - **Hình 3 (Chồng hộp dự đoán của Best FCOS Detector đóng băng):** Nhìn trực tiếp detector bắt đúng/bỏ sót/báo nhầm trên từng miền ảnh giả.
3. **Đo lại đầy đủ cả 2 hệ Metrics trên toàn bộ tập Validation (~1.013 ảnh, chạy chỉ ~2–3 phút trên Kaggle GPU):**
   - **Bảng 1 — Image Generation Metrics trên IR 1 kênh:** `FID` ($\downarrow$), `KID × 10^3` ($\downarrow$), `Edge Structure CosSim` ($\uparrow$), `Thermal Contrast W1` ($\downarrow$), `Person Positive Contrast` ($\uparrow$), `High-Freq Ratio` (soi nhiễu giấu tin steganography), `Ref Val SSIM` & `PSNR`.
   - **Bảng 2 — Frozen Best FCOS Detector & Chất lượng Nhãn giả:** `COCO mAP@[.50:.95]`, `AP50`, `AP75`, `AP_small/medium/large`, `AR100`, `AP50` từng lớp (`person`, `car`, `bicycle`), và `Pseudo-Label Precision / Recall / F1 / Mean IoU` tại `conf = 0.30` & `0.50` so với mốc trần **`Real IR`**.
   - Xuất kèm **biểu đồ cột tổng hợp (`.png`)**, **báo cáo JSON**, và **bảng CSV** để tiện chèn vào báo cáo/slide!

In [ ]:
# ==============================================================================
# CELL 1: CẤU HÌNH ĐƯỜNG DẪN CÁC MODEL CẦN SO SÁNH (CHỈ CẦN CHỈNH TẠI ĐÂY)
# ==============================================================================
CONFIG = {
    # --- 1. Dữ liệu FLIR aligned & Best FCOS Detector (đóng băng làm giám khảo) ---
    "data_root": "/kaggle/input/datasets/tungduong123qwe/flir-dataset/align",
    "best_fcos_checkpoint": "/kaggle/input/datasets/phamduccuong05/fcos-weight/best_model.pth",

    # --- 2. Danh sách các Generator muốn đánh giá & đối chiếu ---
    # Hỗ trợ mọi định dạng: best_net_G_A.pth, latest_net_G_A.pth, epoch_XXX_net_G_A.pth,
    # hoặc cả flsesim_resume_latest.pth / stegogan_resume_latest.pth!
    # (Mẹo: Nếu muốn so cả best và latest, chỉ cần thêm dòng mới vào dict này; để None để tạm bỏ qua 1 model.)
    "generators": {
        "CycleGAN": "/kaggle/input/datasets/bdv212/generator-rgb-ir/latest_net_G_A.pth",
        "F-LSeSim": "/kaggle/input/datasets/phamduccuong05/flsesim-epoch-39/flsesim_resume_latest.pth",
        "StegoGAN": "/kaggle/input/datasets/phamduccuong05/stegogan-epoch-23/stegogan_resume_latest.pth",
    },
    "baseline_name": "CycleGAN",          # Model làm mốc tính độ chênh lệch Δ (+ / -)
    "strict_preflight_check": True,       # Kiểm tra đường dẫn ngay ở Cell 2 (và tự quét /kaggle/input in gợi ý nếu sai)

    # --- 3. Thư mục lưu kết quả đánh giá, ảnh trực quan & biểu đồ ---
    "output_dir": "/kaggle/working/eval_compare_rgb2ir",

    # --- 4. Cấu hình Trực quan hóa bằng mắt thường (Cell 5 & Cell 6) ---
    "num_visual_samples": 8,              # Số cảnh toàn khung hiển thị để soi bằng mắt thường
    "num_zoom_objects": 6,                # Số vật thể nhỏ (ưu tiên bicycle / person) cắt phóng to để soi biên
    "viz_start_index": 0,                 # Đổi số này (vd 8, 16, 24) để xem thêm các nhóm cảnh khác trong tập val
    "viz_fcos_conf_thresh": 0.40,         # Ngưỡng confidence khi vẽ box dự đoán của FCOS lên ảnh ở Hình 3

    # --- 5. Cấu hình Đo Metrics định lượng trên toàn bộ align_validation.txt (Cell 7 & Cell 8) ---
    "eval_max_images": None,              # None = chạy TOÀN BỘ align_validation.txt (~1.013 ảnh); đặt số nguyên (vd 200) nếu muốn test nhanh
    "eval_batch_size": 4,
    "num_workers": 2,
    "fcos_fallback_img_size": 640,        # Dùng nếu best_fcos_checkpoint thuộc dạng training_full_d3t_v2_rtc.py
    "fcos_score_thresh": 0.001,           # Ngưỡng thấp chuẩn COCO để tính chính xác mAP
    "pseudo_label_confs": (0.3, 0.5),     # Các ngưỡng confidence chấm chất lượng nhãn giả (Precision/Recall/F1/IoU)
    "kid_subsets": 50,
    "kid_subset_size": 100,
    "eval_real_ir_upper_bound": True,     # Chấm thêm mốc trần trên Real-IR thật

    "device": "cuda",
    "seed": 42,
}

In [ ]:
# ==============================================================================
# CELL 2: KHỞI TẠO MÔI TRƯỜNG, QUÉT /kaggle/input & ĐỌC TẬP VALIDATION FLIR
# ==============================================================================
import contextlib
import copy
import csv
import gc
import hashlib
import io
import json
import math
import os
import pickle
import random
import subprocess
import sys
import tempfile
import time
import types
import xml.etree.ElementTree as ET
import zipfile
from collections import OrderedDict
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, Subset
import torchvision.transforms.functional as TF

try:
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    HAS_PYCOCOTOOLS = True
except ImportError:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pycocotools"])
        from pycocotools.coco import COCO
        from pycocotools.cocoeval import COCOeval
        HAS_PYCOCOTOOLS = True
    except Exception:
        HAS_PYCOCOTOOLS = False
        print("[INFO] Không cài được pycocotools (môi trường offline); sẽ dùng bộ tính Exact 101-Point COCO AP nội bộ.")


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def scan_kaggle_inputs_hint() -> None:
    kaggle_in = Path("/kaggle/input")
    if not kaggle_in.is_dir():
        return
    print("\n[GỢI Ý ĐƯỜNG DẪN TÌM THẤY TRONG /kaggle/input]:")
    for s in list(kaggle_in.rglob("align_validation.txt"))[:10]:
        print(f"  • data_root gợi ý           : \"{s.parent}\"")
    for p in sorted(kaggle_in.rglob("*.pth"))[:30]:
        if p.is_dir():
            tag = "THƯ MỤC - Kaggle tự giải nén, Notebook tự ghép nạp được!" if any(p.rglob("data.pkl")) else "THƯ MỤC"
            print(f"  • checkpoint (.pth) tìm thấy: \"{p}\"  ({tag})")
        else:
            size_mb = p.stat().st_size / (1024 * 1024)
            print(f"  • checkpoint (.pth) tìm thấy: \"{p}\"  ({size_mb:.1f} MB)")


seed_everything(CONFIG["seed"])
DEVICE = torch.device("cuda" if (CONFIG["device"] == "cuda" and torch.cuda.is_available()) else "cpu")
OUTPUT_DIR = Path(CONFIG["output_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FLIR_CLASSES: Tuple[str, ...] = ("person", "car", "bicycle")
FLIR_CLASS_TO_IDX: Dict[str, int] = {name: idx for idx, name in enumerate(FLIR_CLASSES)}
IGNORE_LABELS = {"FLIR", "dog"}
IR_EXTENSIONS = (".jpeg", ".jpg", ".png")


def resolve_split_file(root: Path, split: str) -> Path:
    filename = f"align_{split}.txt"
    for candidate in (root / filename, root / "ImageSets" / "Main" / filename):
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f"Không tìm thấy '{filename}' trong {root}")


def read_split_stems(split_file: Path) -> List[str]:
    return [line.strip() for line in split_file.read_text(encoding="utf-8").splitlines() if line.strip()]


def ir_stem_to_rgb_path(root: Path, stem: str) -> Optional[Path]:
    base = stem.replace("_PreviewData", "")
    for ext in (".jpg", ".JPG", ".jpeg", ".png"):
        candidate = root / "JPEGImages" / f"{base}_RGB{ext}"
        if candidate.is_file():
            return candidate
    return None


def ir_stem_to_ir_path(root: Path, stem: str) -> Optional[Path]:
    for ext in IR_EXTENSIONS:
        candidate = root / "JPEGImages" / f"{stem}{ext}"
        if candidate.is_file():
            return candidate
    return None


def parse_voc_xml(xml_path: Path, min_area: float = 0.0) -> Tuple[torch.Tensor, torch.Tensor]:
    if not xml_path.is_file():
        return torch.zeros((0, 4), dtype=torch.float32), torch.zeros((0,), dtype=torch.int64)
    root = ET.parse(xml_path).getroot()
    boxes, labels = [], []
    for obj in root.findall("object"):
        name = obj.findtext("name", default="").strip()
        if name in IGNORE_LABELS or name not in FLIR_CLASS_TO_IDX:
            continue
        bnd = obj.find("bndbox")
        if bnd is None:
            continue
        x1 = float(bnd.findtext("xmin"))
        y1 = float(bnd.findtext("ymin"))
        x2 = float(bnd.findtext("xmax"))
        y2 = float(bnd.findtext("ymax"))
        if x2 > x1 and y2 > y1 and (x2 - x1) * (y2 - y1) >= min_area:
            boxes.append([x1, y1, x2, y2])
            labels.append(FLIR_CLASS_TO_IDX[name])
    if not boxes:
        return torch.zeros((0, 4), dtype=torch.float32), torch.zeros((0,), dtype=torch.int64)
    return torch.tensor(boxes, dtype=torch.float32), torch.tensor(labels, dtype=torch.int64)


class FLIRValidationPairDataset(Dataset):
    def __init__(self, root: Path, stems: Sequence[str], min_area: float = 0.0) -> None:
        self.root = root
        self.records = []
        for stem in stems:
            rgb_p = ir_stem_to_rgb_path(root, stem)
            ir_p = ir_stem_to_ir_path(root, stem)
            xml_p = root / "Annotations" / f"{stem}.xml"
            if rgb_p is not None and ir_p is not None and xml_p.is_file():
                self.records.append((stem, rgb_p, ir_p, xml_p))
        if not self.records:
            raise RuntimeError(f"Không tìm thấy mẫu validation hợp lệ trong {root}")
        self.min_area = min_area

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        stem, rgb_p, ir_p, xml_p = self.records[idx]
        rgb_t = TF.to_tensor(Image.open(rgb_p).convert("RGB"))  # [3, H, W] in [0, 1]
        ir_1ch = TF.to_tensor(Image.open(ir_p).convert("L"))    # [1, H, W] in [0, 1]
        boxes, labels = parse_voc_xml(xml_p, min_area=self.min_area)
        return {
            "stem": stem,
            "rgb": rgb_t,
            "ir_1ch": ir_1ch,
            "target": {"boxes": boxes, "labels": labels, "stem": stem},
        }


def val_collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    return {
        "stems": tuple(item["stem"] for item in batch),
        "rgb": [item["rgb"] for item in batch],
        "ir_1ch": [item["ir_1ch"] for item in batch],
        "targets": [item["target"] for item in batch],
    }


gpu_name = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU"
print(f"[ENV] Device = {DEVICE} ({gpu_name}) | PyTorch = {torch.__version__} | pycocotools = {HAS_PYCOCOTOOLS}")

DATA_ROOT = Path(CONFIG["data_root"])
if not DATA_ROOT.is_dir():
    scan_kaggle_inputs_hint()
    if CONFIG.get("strict_preflight_check", True):
        raise FileNotFoundError(f"Không tìm thấy data_root='{DATA_ROOT}'. Hãy chỉnh lại ở Cell 1!")
else:
    VAL_STEMS = read_split_stems(resolve_split_file(DATA_ROOT, "validation"))
    val_eval_ds = FLIRValidationPairDataset(DATA_ROOT, VAL_STEMS)
    print(f"[DATA] Tập Validation FLIR độc lập: {len(val_eval_ds)} ảnh hợp lệ (RGB 3ch + IR 1ch + XML GT).")

In [ ]:
# ==============================================================================
# CELL 3: BỘ NẠP ĐA NĂNG GENERATOR (CYCLEGAN / F-LSeSim / STEGO-GAN / RESUME) & FROZEN FCOS
# ==============================================================================
from torchvision.models.detection import fcos_resnet50_fpn
from torchvision.models.detection.fcos import FCOSHead


# --- 3.1 Bộ unpickle an toàn (không bao giờ lỗi ModuleNotFoundError: No module named 'configs' trên Kaggle!) ---
class _FallbackUnpickledObj:
    def __new__(cls, *args, **kwargs):
        inst = super().__new__(cls)
        inst._args = args
        return inst

    def __init__(self, *args, **kwargs):
        self._args = args
        self.__dict__.update(kwargs)

    def __setstate__(self, state):
        if isinstance(state, dict):
            self.__dict__.update(state)
        else:
            self._state = state

    def get(self, key, default=None):
        return self.__dict__.get(key, default)

    @property
    def value(self):
        return self._args[0] if getattr(self, "_args", ()) else None

    def __hash__(self):
        return hash((type(self).__name__, getattr(self, "_args", ())))

    def __eq__(self, other):
        return isinstance(other, _FallbackUnpickledObj) and getattr(self, "_args", ()) == getattr(other, "_args", ())


class _SafeUnpickler(pickle.Unpickler):
    def find_class(self, mod_name: str, name: str):
        if mod_name == "pathlib" and name in ("PosixPath", "WindowsPath"):
            return Path
        try:
            return super().find_class(mod_name, name)
        except (ModuleNotFoundError, AttributeError):
            return type(f"_Missing_{name}", (_FallbackUnpickledObj,), {"__module__": mod_name})


def is_valid_checkpoint_path(path: str | Path) -> bool:
    """Chấp nhận cả file .pth/.pt chuẩn lẫn thư mục bị Kaggle tự giải nén (chứa data.pkl hoặc .pth bên trong)."""
    p = Path(path)
    if p.is_file():
        return True
    if p.is_dir():
        if any(p.rglob("data.pkl")):
            return True
        if any(c.is_file() for c in p.rglob("*.pth")) or any(c.is_file() for c in p.rglob("*.pt")):
            return True
    return False


def safe_torch_load(path: str | Path, map_location: str | torch.device = "cpu") -> Any:
    """Nạp an toàn cả file .pth chuẩn lẫn thư mục bị Kaggle tự giải nén (chứa data.pkl + data/*)."""
    p = Path(path)
    safe_pickle = types.SimpleNamespace(**{k: getattr(pickle, k) for k in dir(pickle)})
    safe_pickle.Unpickler = _SafeUnpickler

    if p.is_file():
        return torch.load(p, map_location=map_location, weights_only=False, pickle_module=safe_pickle)

    if p.is_dir():
        nested_files = [c for c in sorted(p.rglob("*.pth")) if c.is_file()] + [
            c for c in sorted(p.rglob("*.pt")) if c.is_file()
        ]
        if nested_files:
            print(f"[AUTO-FIND] Phát hiện file checkpoint nằm trong thư mục '{p.name}': {nested_files[0].name}")
            return torch.load(nested_files[0], map_location=map_location, weights_only=False, pickle_module=safe_pickle)

        pkl_candidates = sorted(p.rglob("data.pkl"))
        if pkl_candidates:
            root = pkl_candidates[0].parent
            tmp_fd, tmp_name = tempfile.mkstemp(suffix=".pth")
            os.close(tmp_fd)
            tmp_path = Path(tmp_name)
            try:
                print(f"[AUTO-REPACK] Phát hiện '{p.name}' là thư mục do Kaggle tự giải nén file .pth -> tự ghép tạm để nạp...")
                with zipfile.ZipFile(tmp_path, "w", compression=zipfile.ZIP_STORED) as zf:
                    for f in sorted(root.rglob("*")):
                        if f.is_file():
                            zf.write(f, "archive/" + f.relative_to(root).as_posix())
                return torch.load(tmp_path, map_location=map_location, weights_only=False, pickle_module=safe_pickle)
            finally:
                if tmp_path.exists():
                    tmp_path.unlink()

    raise FileNotFoundError(
        f"Không tìm thấy file checkpoint (.pth) hợp lệ hoặc thư mục PyTorch (data.pkl) tại: {p}"
    )


# --- 3.2 Kiến trúc Generator 3ch -> 1ch (khớp 100% CycleGAN, F-LSeSim & nhánh sạch StegoGAN) ---
class _ResidualBlock(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.ReflectionPad2d(1), nn.Conv2d(channels, channels, 3, bias=True),
            nn.InstanceNorm2d(channels), nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1), nn.Conv2d(channels, channels, 3, bias=True),
            nn.InstanceNorm2d(channels),
        )

    def forward(self, x: Tensor) -> Tensor:
        return x + self.conv_block(x)


class CycleGANGenerator(nn.Module):
    """Khớp 100% models/cyclegan_generator.py (RGB 3ch -> IR 1ch)."""

    def __init__(self, ngf: int = 64):
        super().__init__()
        layers: List[nn.Module] = [
            nn.ReflectionPad2d(3), nn.Conv2d(3, ngf, 7, bias=True),
            nn.InstanceNorm2d(ngf), nn.ReLU(inplace=True),
        ]
        channels = ngf
        for _ in range(2):
            layers.extend([
                nn.Conv2d(channels, channels * 2, 3, stride=2, padding=1, bias=True),
                nn.InstanceNorm2d(channels * 2), nn.ReLU(inplace=True),
            ])
            channels *= 2
        layers.extend(_ResidualBlock(channels) for _ in range(9))
        for _ in range(2):
            layers.extend([
                nn.ConvTranspose2d(channels, channels // 2, 3, stride=2, padding=1, output_padding=1, bias=True),
                nn.InstanceNorm2d(channels // 2), nn.ReLU(inplace=True),
            ])
            channels //= 2
        layers.extend([nn.ReflectionPad2d(3), nn.Conv2d(ngf, 1, 7, bias=True), nn.Tanh()])
        self.model = nn.Sequential(*layers)

    def forward(self, images: Tensor) -> Tensor:
        return self.model(images)


class FrozenGANTranslator(nn.Module):
    """Khớp 100% training/gan_bridge.py: translate_1ch trả [B,1,H,W] trong [0,1]; forward lặp 3 kênh cho FCOS."""

    def __init__(self, generator: nn.Module, info_tag: str = ""):
        super().__init__()
        self.generator = generator
        self.info_tag = info_tag
        self.requires_grad_(False)
        self.eval()

    def train(self, mode: bool = True):
        return super().train(False)

    @torch.no_grad()
    def translate_1ch(self, images_rgb_01: Tensor) -> Tensor:
        if images_rgb_01.ndim != 4 or images_rgb_01.shape[1] != 3:
            raise ValueError(f"Input phải có dạng [B, 3, H, W], nhận được {tuple(images_rgb_01.shape)}")
        height, width = images_rgb_01.shape[-2:]
        pad_h = max(8, (height + 3) // 4 * 4) - height
        pad_w = max(8, (width + 3) // 4 * 4) - width
        outputs_1ch = []
        for image in images_rgb_01.split(1):
            gan_input = F.pad(image * 2.0 - 1.0, (0, pad_w, 0, pad_h), mode="replicate")
            raw_out = self.generator(gan_input)
            if raw_out.shape != (1, 1, height + pad_h, width + pad_w):
                raise ValueError(f"Generator bắt buộc xuất 1 kênh [1,1,H,W], nhận được {tuple(raw_out.shape)}")
            thermal_1ch = ((raw_out[:, :, :height, :width] + 1.0) * 0.5).clamp(0.0, 1.0)
            outputs_1ch.append(thermal_1ch)
        return torch.cat(outputs_1ch, dim=0)

    @torch.no_grad()
    def forward(self, images_rgb_01: Tensor) -> Tensor:
        return self.translate_1ch(images_rgb_01).repeat(1, 3, 1, 1)


def _convert_stego_native_ga_to_flat_state(stego_ga_state: Dict[str, Tensor]) -> OrderedDict:
    """Tự động chuyển đổi state_dict của StegoResnetMaskV1Generator (conv1/conv2/conv3/decoder) sang khóa phẳng CycleGANGenerator."""
    def _prefix_keys(prefix: str) -> Dict[int, Dict[str, Tensor]]:
        grouped: Dict[int, Dict[str, Tensor]] = {}
        for k, v in stego_ga_state.items():
            if not k.startswith(prefix + "."):
                continue
            rest = k[len(prefix) + 1 :]
            idx_str, sub = rest.split(".", 1)
            grouped.setdefault(int(idx_str), {})[sub] = v
        return grouped

    # Đếm số khối ResNet nằm trong conv2 và conv3
    conv2_indices = sorted(_prefix_keys("conv2").keys())
    conv3_indices = sorted(_prefix_keys("conv3").keys())
    total_res = len(conv2_indices) + len(conv3_indices)
    if total_res != 9:
        raise ValueError(f"StegoGAN native netG_A cần đúng 9 khối ResNet, tìm thấy {total_res} khối.")

    flat: OrderedDict[str, Tensor] = OrderedDict()
    # conv1 -> indices 0..9 trong CycleGANGenerator.model (Conv2d nằm ở 1, 4, 7)
    for k, v in stego_ga_state.items():
        if k.startswith("conv1."):
            flat["model." + k[len("conv1."):]] = v
    # conv2 -> indices 10 .. 10+len(conv2)-1
    for i, old_idx in enumerate(conv2_indices):
        for sub, v in _prefix_keys("conv2")[old_idx].items():
            flat[f"model.{10 + i}.{sub}"] = v
    # conv3 -> tiếp nối ngay sau conv2
    for j, old_idx in enumerate(conv3_indices):
        for sub, v in _prefix_keys("conv3")[old_idx].items():
            flat[f"model.{10 + len(conv2_indices) + j}.{sub}"] = v
    # decoder -> indices 19..27 (ConvTranspose2d ở 0, 3; Conv2d cuối ở 7 -> 19+0=19, 19+3=22, 19+7=26)
    for k, v in stego_ga_state.items():
        if k.startswith("decoder."):
            rest = k[len("decoder."):]
            idx_str, sub = rest.split(".", 1)
            flat[f"model.{19 + int(idx_str)}.{sub}"] = v
    return flat


def load_any_rgb2ir_generator(checkpoint_path: str | Path, device: torch.device) -> FrozenGANTranslator:
    """
    Nạp linh hoạt MỌI định dạng trọng số RGB(3ch) -> IR(1ch):
      1) best_net_G_A.pth / latest_net_G_A.pth / epoch_XXX_net_G_A.pth (CycleGANGenerator)
      2) flsesim_resume_latest.pth (chứa khóa 'netG' + 'epoch')
      3) stegogan_resume_latest.pth (chứa khóa 'netG_A' dạng conv1/conv2/decoder + 'epoch')
      4) stego_native_netG_A.pth (trực tiếp conv1/conv2/decoder)
    """
    path = Path(checkpoint_path)
    if not is_valid_checkpoint_path(path):
        raise FileNotFoundError(f"Không tìm thấy checkpoint generator: {path}")

    raw = safe_torch_load(path)
    tag = path.name
    state = raw

    if isinstance(raw, dict):
        if "netG" in raw and isinstance(raw["netG"], dict):
            ep = raw.get("epoch", "?")
            tag = f"{path.name} [F-LSeSim resume @ epoch {ep}]"
            state = raw["netG"]
        elif "netG_A" in raw and isinstance(raw["netG_A"], dict):
            ep = raw.get("epoch", "?")
            tag = f"{path.name} [StegoGAN resume @ epoch {ep}]"
            state = raw["netG_A"]
        elif "state_dict" in raw and isinstance(raw["state_dict"], dict):
            state = raw["state_dict"]

    if not isinstance(state, dict) or not state:
        raise ValueError(f"File '{path}' không chứa state_dict hợp lệ.")

    if all(k.startswith("module.") for k in state):
        state = {k.removeprefix("module."): v for k, v in state.items()}

    # Nếu là StegoResnetMaskV1Generator dạng tách (conv1/conv2/decoder), tự chuyển sang khóa phẳng CycleGANGenerator
    if any(k.startswith("conv1.") for k in state) and any(k.startswith("decoder.") for k in state):
        state = _convert_stego_native_ga_to_flat_state(state)

    gen = CycleGANGenerator(ngf=64)
    gen.load_state_dict(state, strict=True)
    return FrozenGANTranslator(gen, info_tag=tag).to(device)


# --- 3.3 Bộ nạp Đóng băng Best FCOS Detector (hỗ trợ cả 2 định dạng FCOS của dự án) ---
class _RegressionHeadWithoutCenterness(nn.Module):
    def __init__(self, conv: nn.Module, bbox_reg: nn.Module, num_anchors: int):
        super().__init__()
        self.conv = conv
        self.bbox_reg = bbox_reg
        self.num_anchors = num_anchors

    def forward(self, features: Sequence[Tensor]) -> Tensor:
        outputs = []
        for feature in features:
            reg = F.relu(self.bbox_reg(self.conv(feature)))
            b, _, h, w = reg.shape
            outputs.append(reg.view(b, self.num_anchors, 4, h, w).permute(0, 3, 4, 1, 2).reshape(b, -1, 4))
        return torch.cat(outputs, dim=1)


class _StandaloneFCOSIoUHead(nn.Module):
    def __init__(self, base_head: FCOSHead, num_classes: int = 3):
        super().__init__()
        base_cls = base_head.classification_head
        base_reg = base_head.regression_head
        in_channels = next(m.in_channels for m in base_cls.conv if isinstance(m, nn.Conv2d))
        num_anchors = base_cls.num_anchors
        cls_head = copy.deepcopy(base_cls)
        cls_head.num_classes = num_classes
        cls_head.cls_logits = nn.Conv2d(in_channels, num_anchors * num_classes, kernel_size=3, stride=1, padding=1)

        self.classification_head = cls_head
        self.regression_head = _RegressionHeadWithoutCenterness(
            conv=copy.deepcopy(base_reg.conv),
            bbox_reg=copy.deepcopy(base_reg.bbox_reg),
            num_anchors=num_anchors,
        )
        self.quality_predictor = nn.Conv2d(in_channels, num_anchors, kernel_size=3, stride=1, padding=1)
        self.num_classes = num_classes

    @staticmethod
    def _flatten(out: Tensor, c_per_anchor: int) -> Tensor:
        b, _, h, w = out.shape
        return out.view(b, -1, c_per_anchor, h, w).permute(0, 3, 4, 1, 2).reshape(b, -1, c_per_anchor)

    def forward(self, features: Sequence[Tensor]) -> Dict[str, Tensor]:
        cls_outs, qual_outs = [], []
        for f in features:
            cf = self.classification_head.conv(f)
            cls_outs.append(self._flatten(self.classification_head.cls_logits(cf), self.num_classes))
            qual_outs.append(self._flatten(self.quality_predictor(cf), 1))
        return {
            "cls_logits": torch.cat(cls_outs, dim=1),
            "bbox_regression": self.regression_head(features),
            "quality_logits": torch.cat(qual_outs, dim=1),
        }


class _StandaloneIoUFCOSInference(nn.Module):
    def __init__(self, detector: nn.Module):
        super().__init__()
        self.detector = detector

    @torch.no_grad()
    def forward(self, images: List[Tensor]) -> List[Dict[str, Tensor]]:
        orig_sizes = [(int(img.shape[-2]), int(img.shape[-1])) for img in images]
        transformed, _ = self.detector.transform(images, None)
        feats = self.detector.backbone(transformed.tensors)
        if isinstance(feats, Tensor):
            feats = OrderedDict([("0", feats)])
        feat_list = list(feats.values())
        head_out = self.detector.head(feat_list)
        anchors_list = self.detector.anchor_generator(transformed, feat_list)
        anchors_per_loc = self.detector.anchor_generator.num_anchors_per_location()
        num_anchors_per_level = [
            int(f.shape[-2] * f.shape[-1] * anchors_per_loc[lvl])
            for lvl, f in enumerate(feat_list)
        ]
        anchors_per_level = [list(a.split(num_anchors_per_level, dim=0)) for a in anchors_list]
        per_level = {k: list(v.split(num_anchors_per_level, dim=1)) for k, v in head_out.items()}
        native = self.detector.postprocess_detections(
            {
                "cls_logits": per_level["cls_logits"],
                "bbox_regression": per_level["bbox_regression"],
                "bbox_ctrness": per_level["quality_logits"],
            },
            anchors_per_level,
            list(transformed.image_sizes),
        )
        return self.detector.transform.postprocess(native, list(transformed.image_sizes), orig_sizes)


class FrozenFCOSJudge(nn.Module):
    def __init__(self, predictor: nn.Module, fmt_name: str, fixed_resize: Optional[int] = None):
        super().__init__()
        self.predictor = predictor
        self.fmt_name = fmt_name
        self.fixed_resize = fixed_resize
        self.requires_grad_(False)
        self.eval()

    def train(self, mode: bool = True):
        return super().train(False)

    @torch.no_grad()
    def predict(self, images_3ch_01: List[Tensor]) -> List[Dict[str, Tensor]]:
        if self.fixed_resize is not None:
            resized_imgs, scales = [], []
            for img in images_3ch_01:
                h, w = img.shape[-2:]
                scales.append((w / self.fixed_resize, h / self.fixed_resize))
                resized_imgs.append(
                    F.interpolate(img.unsqueeze(0), size=(self.fixed_resize, self.fixed_resize), mode="bilinear", align_corners=False).squeeze(0)
                )
            preds = self.predictor(resized_imgs)
            restored = []
            for pred, (sx, sy) in zip(preds, scales):
                boxes = pred["boxes"].clone()
                if boxes.numel() > 0:
                    boxes[:, [0, 2]] *= sx
                    boxes[:, [1, 3]] *= sy
                restored.append({"boxes": boxes, "scores": pred["scores"], "labels": pred["labels"]})
            return restored
        return self.predictor(images_3ch_01)


def load_frozen_fcos_detector(
    checkpoint_path: str | Path,
    device: torch.device,
    *,
    score_thresh: float = 0.001,
    fallback_img_size: int = 640,
) -> FrozenFCOSJudge:
    path = Path(checkpoint_path)
    if not is_valid_checkpoint_path(path):
        raise FileNotFoundError(f"Không tìm thấy best_fcos_checkpoint tại: {path}")

    raw = safe_torch_load(path)

    if isinstance(raw, dict) and "student" in raw and any(k.startswith("adapter.detector.") for k in raw["student"]):
        meta = raw.get("metadata", {})
        if not isinstance(meta, dict):
            meta = vars(meta) if hasattr(meta, "__dict__") else {}
        saved_m = meta.get("model")
        if saved_m is None:
            cfg_obj = meta.get("config", {})
            if not isinstance(cfg_obj, dict):
                cfg_obj = vars(cfg_obj) if hasattr(cfg_obj, "__dict__") else {}
            saved_m = cfg_obj.get("model", {})
        if not isinstance(saved_m, dict):
            saved_m = vars(saved_m) if hasattr(saved_m, "__dict__") else {}

        num_classes = int(saved_m.get("num_classes", 3))
        raw_min = saved_m.get("min_sizes", (800,))
        min_sizes = (int(raw_min),) if isinstance(raw_min, (int, float)) else tuple(int(v) for v in raw_min)
        max_size = int(saved_m.get("max_size", 1333))
        nms_thresh = float(saved_m.get("nms_thresh", 0.6))
        topk = int(saved_m.get("topk_candidates", 1000))

        base = fcos_resnet50_fpn(
            weights=None,
            weights_backbone=None,
            num_classes=91,
            min_size=(min_sizes[-1],),
            max_size=max_size,
            score_thresh=score_thresh,
            nms_thresh=nms_thresh,
            topk_candidates=topk,
            detections_per_img=100,
        )
        base.head = _StandaloneFCOSIoUHead(base.head, num_classes=num_classes)
        det_state = {
            k.removeprefix("adapter.detector."): v
            for k, v in raw["student"].items()
            if k.startswith("adapter.detector.")
        }
        base.load_state_dict(det_state, strict=True)
        return FrozenFCOSJudge(_StandaloneIoUFCOSInference(base), fmt_name="DAOD-Adapter-FCOSIoUHead", fixed_resize=None).to(device)

    state = raw
    for key in ("model", "s_model", "state_dict", "student"):
        if isinstance(state, dict) and key in state and isinstance(state[key], dict):
            state = state[key]
            break
    if all(k.startswith("module.") for k in state):
        state = {k.removeprefix("module."): v for k, v in state.items()}

    base_std = fcos_resnet50_fpn(
        weights=None,
        weights_backbone=None,
        num_classes=len(FLIR_CLASSES),
        min_size=fallback_img_size,
        max_size=fallback_img_size,
        score_thresh=score_thresh,
        detections_per_img=100,
    )
    base_std.load_state_dict(state, strict=True)
    return FrozenFCOSJudge(base_std, fmt_name=f"Torchvision-FCOS-{fallback_img_size}", fixed_resize=fallback_img_size).to(device)


# --- 3.4 Nạp & Xác minh toàn bộ Generator + Best FCOS ngay lập tức (Preflight Check) ---
LOADED_TRANSLATORS: Dict[str, FrozenGANTranslator] = OrderedDict()
FROZEN_FCOS_JUDGE: Optional[FrozenFCOSJudge] = None

if DATA_ROOT.is_dir():
    errors = []
    for name, ckpt_path in CONFIG["generators"].items():
        if not ckpt_path:
            continue
        if not is_valid_checkpoint_path(ckpt_path):
            errors.append(f"Generator '{name}': không tìm thấy file/thư mục checkpoint hợp lệ '{ckpt_path}'")
        else:
            trans = load_any_rgb2ir_generator(ckpt_path, DEVICE)
            LOADED_TRANSLATORS[name] = trans
            print(f"[LOADED GENERATOR] {name:16s} <- {trans.info_tag}")

    fcos_path = CONFIG.get("best_fcos_checkpoint")
    if fcos_path:
        if not is_valid_checkpoint_path(fcos_path):
            errors.append(f"Best FCOS Detector: không tìm thấy file/thư mục checkpoint hợp lệ '{fcos_path}'")
        else:
            FROZEN_FCOS_JUDGE = load_frozen_fcos_detector(
                fcos_path,
                DEVICE,
                score_thresh=CONFIG["fcos_score_thresh"],
                fallback_img_size=CONFIG["fcos_fallback_img_size"],
            )
            print(f"[LOADED FCOS JUDGE] {Path(fcos_path).name} ({FROZEN_FCOS_JUDGE.fmt_name}) — ĐÃ ĐÓNG BĂNG!")

    if not LOADED_TRANSLATORS:
        errors.append("Chưa nạp được generator nào trong CONFIG['generators']!")

    if errors:
        scan_kaggle_inputs_hint()
        if CONFIG.get("strict_preflight_check", True):
            raise FileNotFoundError(
                "[PREFLIGHT LỖI ĐƯỜNG DẪN]:\n  - " + "\n  - ".join(errors) +
                "\nHãy copy đường dẫn tìm thấy bên trên vào Cell 1 (CONFIG)!"
            )
        else:
            print("[CẢNH BÁO]", errors)

In [ ]:
# ==============================================================================
# CELL 4: BỘ TÍNH 2 HỆ METRICS (GENERATION METRICS TRÊN IR 1 KÊNH + FROZEN FCOS & PSEUDO-LABEL)
# ==============================================================================
from torchvision.models import Inception_V3_Weights, inception_v3


def box_iou_matrix(boxes1: Tensor, boxes2: Tensor) -> Tensor:
    if boxes1.numel() == 0 or boxes2.numel() == 0:
        return torch.zeros((boxes1.shape[0], boxes2.shape[0]), dtype=torch.float32)
    area1 = (boxes1[:, 2] - boxes1[:, 0]).clamp_min(0) * (boxes1[:, 3] - boxes1[:, 1]).clamp_min(0)
    area2 = (boxes2[:, 2] - boxes2[:, 0]).clamp_min(0) * (boxes2[:, 3] - boxes2[:, 1]).clamp_min(0)
    lt = torch.maximum(boxes1[:, None, :2], boxes2[None, :, :2])
    rb = torch.minimum(boxes1[:, None, 2:], boxes2[None, :, 2:])
    wh = (rb - lt).clamp_min(0)
    inter = wh[:, :, 0] * wh[:, :, 1]
    return inter / (area1[:, None] + area2[None, :] - inter).clamp_min(1e-7)


def compute_pseudo_label_quality(
    predictions: Sequence[Dict[str, Tensor]],
    targets: Sequence[Dict[str, Tensor]],
    conf_thresh: float = 0.5,
    iou_thresh: float = 0.5,
) -> Dict[str, float]:
    tp, fp, total_gt = 0, 0, 0
    matched_ious: List[float] = []
    for pred, tgt in zip(predictions, targets):
        gt_boxes = tgt["boxes"].cpu()
        gt_labels = tgt["labels"].cpu()
        total_gt += int(gt_boxes.shape[0])

        keep = pred["scores"].cpu() >= conf_thresh
        p_boxes = pred["boxes"].cpu()[keep]
        p_scores = pred["scores"].cpu()[keep]
        p_labels = pred["labels"].cpu()[keep]
        if p_boxes.shape[0] > 0:
            order = torch.argsort(p_scores, descending=True)
            p_boxes, p_labels = p_boxes[order], p_labels[order]

        gt_used = torch.zeros((gt_boxes.shape[0],), dtype=torch.bool)
        ious = box_iou_matrix(p_boxes, gt_boxes)
        for i in range(p_boxes.shape[0]):
            same_cls = (gt_labels == p_labels[i]) & (~gt_used)
            if bool(same_cls.any()):
                cand_ious = ious[i].clone()
                cand_ious[~same_cls] = -1.0
                best_iou, best_j = cand_ious.max(dim=0)
                if float(best_iou) >= iou_thresh:
                    tp += 1
                    gt_used[best_j] = True
                    matched_ious.append(float(best_iou))
                else:
                    fp += 1
            else:
                fp += 1

    precision = tp / max(1, tp + fp)
    recall = tp / max(1, total_gt)
    f1 = 2.0 * precision * recall / max(1e-8, precision + recall)
    return {
        "conf_thresh": conf_thresh,
        "PL_Precision": float(precision),
        "PL_Recall": float(recall),
        "PL_F1": float(f1),
        "PL_MeanMatchedIoU": float(np.mean(matched_ious)) if matched_ious else 0.0,
        "PL_BoxesPerImg": float((tp + fp) / max(1, len(targets))),
        "TP": int(tp),
        "FP": int(fp),
        "GT": int(total_gt),
    }


def _fallback_coco_ap(predictions: Sequence[Dict[str, Tensor]], targets: Sequence[Dict[str, Tensor]]) -> Dict[str, float]:
    iou_thrs = np.linspace(0.5, 0.95, 10)
    recall_thrs = np.linspace(0.0, 1.0, 101)
    ap_matrix = np.full((len(iou_thrs), len(FLIR_CLASSES)), np.nan, dtype=np.float64)

    for c_idx in range(len(FLIR_CLASSES)):
        gt_by_img = {}
        total_gt = 0
        det_records = []
        for img_id, (pred, tgt) in enumerate(zip(predictions, targets)):
            g_mask = tgt["labels"].cpu() == c_idx
            g_boxes = tgt["boxes"].cpu()[g_mask]
            gt_by_img[img_id] = g_boxes
            total_gt += int(g_boxes.shape[0])

            p_mask = pred["labels"].cpu() == c_idx
            p_boxes = pred["boxes"].cpu()[p_mask]
            p_scores = pred["scores"].cpu()[p_mask]
            for b, s in zip(p_boxes, p_scores):
                det_records.append((float(s), img_id, b.unsqueeze(0)))

        if total_gt == 0:
            continue
        det_records.sort(key=lambda x: x[0], reverse=True)
        for t_idx, thr in enumerate(iou_thrs):
            used = {img_id: np.zeros(len(boxes), dtype=bool) for img_id, boxes in gt_by_img.items()}
            tps = np.zeros(len(det_records), dtype=np.float64)
            fps = np.zeros(len(det_records), dtype=np.float64)
            for d_i, (_, img_id, box_1) in enumerate(det_records):
                g_boxes = gt_by_img[img_id]
                if len(g_boxes) == 0:
                    fps[d_i] = 1.0
                    continue
                ious = box_iou_matrix(box_1, g_boxes)[0].numpy()
                ious[used[img_id]] = -1.0
                j = int(np.argmax(ious))
                if ious[j] >= thr:
                    tps[d_i] = 1.0
                    used[img_id][j] = True
                else:
                    fps[d_i] = 1.0
            tp_cum = np.cumsum(tps)
            fp_cum = np.cumsum(fps)
            rec = tp_cum / total_gt
            prec = tp_cum / np.maximum(tp_cum + fp_cum, 1e-9)
            for i in range(len(prec) - 2, -1, -1):
                prec[i] = max(prec[i], prec[i + 1])
            inds = np.searchsorted(rec, recall_thrs, side="left")
            q = np.array([prec[idx] if idx < len(prec) else 0.0 for idx in inds])
            ap_matrix[t_idx, c_idx] = float(np.mean(q))

    out = {
        "AP": float(np.nanmean(ap_matrix)),
        "AP50": float(np.nanmean(ap_matrix[0])),
        "AP75": float(np.nanmean(ap_matrix[5])),
    }
    for c_idx, name in enumerate(FLIR_CLASSES):
        out[f"AP/{name}"] = float(np.nanmean(ap_matrix[:, c_idx]))
        out[f"AP50/{name}"] = float(ap_matrix[0, c_idx])
        out[f"AP75/{name}"] = float(ap_matrix[5, c_idx])
    return out


def compute_coco_detection_metrics(
    predictions: Sequence[Dict[str, Tensor]], targets: Sequence[Dict[str, Tensor]]
) -> Dict[str, float]:
    if not HAS_PYCOCOTOOLS:
        return _fallback_coco_ap(predictions, targets)

    categories = [{"id": i + 1, "name": name} for i, name in enumerate(FLIR_CLASSES)]
    images = [{"id": i + 1} for i in range(len(targets))]
    annotations, detections = [], []
    ann_id = 1
    for img_idx, (pred, tgt) in enumerate(zip(predictions, targets), start=1):
        for box, lbl in zip(tgt["boxes"].cpu().tolist(), tgt["labels"].cpu().tolist()):
            x1, y1, x2, y2 = box
            w, h = x2 - x1, y2 - y1
            annotations.append({
                "id": ann_id, "image_id": img_idx, "category_id": int(lbl) + 1,
                "bbox": [x1, y1, w, h], "area": w * h, "iscrowd": 0,
            })
            ann_id += 1
        for box, score, lbl in zip(
            pred["boxes"].cpu().tolist(), pred["scores"].cpu().tolist(), pred["labels"].cpu().tolist()
        ):
            x1, y1, x2, y2 = box
            detections.append({
                "image_id": img_idx, "category_id": int(lbl) + 1,
                "bbox": [x1, y1, x2 - x1, y2 - y1], "score": float(score),
            })

    with contextlib.redirect_stdout(io.StringIO()):
        coco_gt = COCO()
        coco_gt.dataset = {"info": {}, "licenses": [], "images": images, "annotations": annotations, "categories": categories}
        coco_gt.createIndex()
        if detections:
            coco_dt = coco_gt.loadRes(detections)
        else:
            coco_dt = COCO()
            coco_dt.dataset = {"images": images, "categories": categories, "annotations": []}
            coco_dt.createIndex()

        coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
        coco_eval.params.catIds = [1, 2, 3]
        coco_eval.params.imgIds = list(range(1, len(targets) + 1))
        coco_eval.evaluate()
        coco_eval.accumulate()
        coco_eval.summarize()

    stat_names = ("AP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR1", "AR10", "AR100")
    metrics = {name: float(coco_eval.stats[i]) for i, name in enumerate(stat_names)}
    precision = coco_eval.eval["precision"]
    area_idx = list(coco_eval.params.areaRngLbl).index("all")
    max_det_idx = len(coco_eval.params.maxDets) - 1
    iou_thrs = np.asarray(coco_eval.params.iouThrs)
    for c_idx, c_name in enumerate(FLIR_CLASSES):
        cp = precision[:, :, c_idx, area_idx, max_det_idx]
        valid = cp[cp > -1]
        metrics[f"AP/{c_name}"] = float(valid.mean()) if valid.size else float("nan")
        for thr, tag in ((0.5, "AP50"), (0.75, "AP75")):
            m_idx = np.flatnonzero(np.isclose(iou_thrs, thr))
            v = cp[m_idx[0]][cp[m_idx[0]] > -1]
            metrics[f"{tag}/{c_name}"] = float(v.mean()) if v.size else float("nan")
    return metrics


class InceptionPool3Extractor(nn.Module):
    def __init__(self):
        super().__init__()
        try:
            net = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1, transform_input=False)
        except Exception as err:
            raise RuntimeError(
                "Không tải được InceptionV3 ImageNet weights (hãy bật 'Internet: On' trong Settings của Kaggle!)."
            ) from err
        net.fc = nn.Identity()
        net.aux_logits = False
        self.net = net
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        self.requires_grad_(False)
        self.eval()

    @torch.no_grad()
    def forward(self, ir_1ch_01: Tensor) -> Tensor:
        assert ir_1ch_01.shape[1] == 1
        x = F.interpolate(ir_1ch_01.repeat(1, 3, 1, 1), size=(299, 299), mode="bilinear", align_corners=False)
        x = (x - self.mean) / self.std
        return self.net(x)


def compute_fid_and_kid(
    real_feats: np.ndarray,
    fake_feats: np.ndarray,
    *,
    kid_subsets: int = 50,
    kid_subset_size: int = 100,
    seed: int = 42,
) -> Dict[str, float]:
    r64 = torch.from_numpy(np.asarray(real_feats, dtype=np.float64))
    f64 = torch.from_numpy(np.asarray(fake_feats, dtype=np.float64))

    mu1, mu2 = r64.mean(dim=0), f64.mean(dim=0)
    r_c, f_c = r64 - mu1, f64 - mu2
    sigma1 = (r_c.T @ r_c) / max(1, r64.shape[0] - 1)
    sigma2 = (f_c.T @ f_c) / max(1, f64.shape[0] - 1)

    eps_eye = torch.eye(sigma1.shape[0], dtype=torch.float64) * 1e-6
    eigvals1, eigvecs1 = torch.linalg.eigh(sigma1 + eps_eye)
    sqrt_sigma1 = (eigvecs1 * torch.sqrt(eigvals1.clamp_min(0.0))) @ eigvecs1.T
    prod_sym = sqrt_sigma1 @ (sigma2 + eps_eye) @ sqrt_sigma1
    prod_sym = 0.5 * (prod_sym + prod_sym.T)
    trace_covmean = float(torch.sqrt(torch.linalg.eigvalsh(prod_sym).clamp_min(0.0)).sum().item())
    diff = mu1 - mu2
    fid = float((diff.dot(diff) + torch.trace(sigma1) + torch.trace(sigma2) - 2.0 * trace_covmean).clamp_min(0.0).item())

    r_np = r64.numpy()
    f_np = f64.numpy()
    rng = np.random.default_rng(seed)
    m = min(kid_subset_size, len(r_np), len(f_np))
    d = float(r_np.shape[1])
    mmds = []
    for _ in range(kid_subsets):
        x = r_np[rng.choice(len(r_np), m, replace=False)]
        y = f_np[rng.choice(len(f_np), m, replace=False)]
        k_xx = (x @ x.T / d + 1.0) ** 3
        k_yy = (y @ y.T / d + 1.0) ** 3
        k_xy = (x @ y.T / d + 1.0) ** 3
        mmd = (
            (k_xx.sum() - np.trace(k_xx)) / max(1, m * (m - 1))
            + (k_yy.sum() - np.trace(k_yy)) / max(1, m * (m - 1))
            - 2.0 * k_xy.mean()
        )
        mmds.append(float(mmd))
    return {
        "FID": fid,
        "KID_x1000_mean": float(np.mean(mmds) * 1000.0),
        "KID_x1000_std": float(np.std(mmds) * 1000.0),
    }


def _sobel_magnitude(img_1ch: Tensor) -> Tensor:
    kx = torch.tensor([[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]], device=img_1ch.device).view(1, 1, 3, 3)
    ky = torch.tensor([[-1.0, -2.0, -1.0], [0.0, 0.0, 0.0], [1.0, 2.0, 1.0]], device=img_1ch.device).view(1, 1, 3, 3)
    gx = F.conv2d(F.pad(img_1ch, (1, 1, 1, 1), mode="replicate"), kx)
    gy = F.conv2d(F.pad(img_1ch, (1, 1, 1, 1), mode="replicate"), ky)
    return torch.sqrt(gx * gx + gy * gy + 1e-8)


def _laplacian_energy(img_1ch: Tensor) -> float:
    klap = torch.tensor([[0.0, -1.0, 0.0], [-1.0, 4.0, -1.0], [0.0, -1.0, 0.0]], device=img_1ch.device).view(1, 1, 3, 3)
    resp = F.conv2d(F.pad(img_1ch, (1, 1, 1, 1), mode="replicate"), klap)
    return float((resp ** 2).mean().item())


def _ssim_1ch(x: Tensor, y: Tensor) -> float:
    c1, c2 = 0.01 ** 2, 0.03 ** 2
    mu_x = F.avg_pool2d(x, 11, 1, 5)
    mu_y = F.avg_pool2d(y, 11, 1, 5)
    sigma_x = F.avg_pool2d(x * x, 11, 1, 5) - mu_x ** 2
    sigma_y = F.avg_pool2d(y * y, 11, 1, 5) - mu_y ** 2
    sigma_xy = F.avg_pool2d(x * y, 11, 1, 5) - mu_x * mu_y
    ssim_map = ((2 * mu_x * mu_y + c1) * (2 * sigma_xy + c2)) / ((mu_x ** 2 + mu_y ** 2 + c1) * (sigma_x + sigma_y + c2))
    return float(ssim_map.mean().item())


def _signed_box_ring_contrasts(img_1ch_2d: Tensor, boxes: Tensor, labels: Tensor) -> List[Tuple[int, float]]:
    H, W = img_1ch_2d.shape
    results = []
    for box, lbl in zip(boxes.tolist(), labels.tolist()):
        x1, y1, x2, y2 = box
        cx, cy = 0.5 * (x1 + x2), 0.5 * (y1 + y2)
        bw, bh = x2 - x1, y2 - y1
        if bw < 6 or bh < 6:
            continue
        ix1, iy1 = max(0, int(round(cx - 0.4 * bw))), max(0, int(round(cy - 0.4 * bh)))
        ix2, iy2 = min(W, int(round(cx + 0.4 * bw))), min(H, int(round(cy + 0.4 * bh)))
        ox1, oy1 = max(0, int(round(cx - 0.65 * bw))), max(0, int(round(cy - 0.65 * bh)))
        ox2, oy2 = min(W, int(round(cx + 0.65 * bw))), min(H, int(round(cy + 0.65 * bh)))
        if ix2 <= ix1 + 1 or iy2 <= iy1 + 1:
            continue
        core = img_1ch_2d[iy1:iy2, ix1:ix2].reshape(-1)
        outer_patch = img_1ch_2d[oy1:oy2, ox1:ox2]
        mask = torch.ones_like(outer_patch, dtype=torch.bool)
        mask[iy1 - oy1 : iy2 - oy1, ix1 - ox1 : ix2 - ox1] = False
        ring = outer_patch[mask]
        if core.numel() < 4 or ring.numel() < 8:
            continue
        med_obj = float(core.median().item())
        med_bg = float(ring.median().item())
        mad_bg = float((ring - med_bg).abs().median().item())
        c_val = (med_obj - med_bg) / (1.4826 * mad_bg + 1e-4)
        results.append((int(lbl), float(np.clip(c_val, -15.0, 15.0))))
    return results


def _wasserstein_1d(u: Sequence[float], v: Sequence[float]) -> float:
    if not u or not v:
        return float("nan")
    u_s = np.sort(np.asarray(u, dtype=np.float64))
    v_s = np.sort(np.asarray(v, dtype=np.float64))
    grid = np.linspace(0.0, 1.0, 256)
    return float(np.mean(np.abs(np.quantile(u_s, grid) - np.quantile(v_s, grid))))

In [ ]:
# ==============================================================================
# CELL 5: ĐÁNH GIÁ CHẤT LƯỢNG ẢNH IR SINH RA BẰNG MẮT THƯỜNG (TOÀN KHUNG + ZOOM VẬT THỂ NHỎ)
# ==============================================================================
def select_diverse_visual_indices(ds: FLIRValidationPairDataset, count: int, start_offset: int = 0) -> List[int]:
    """Chọn các cảnh validation đa dạng (ưu tiên có person/bicycle + độ sáng nền khác nhau)."""
    preferred, fallback = [], []
    for i in range(len(ds)):
        lbls = set(ds[i]["target"]["labels"].tolist())
        if FLIR_CLASS_TO_IDX["bicycle"] in lbls or FLIR_CLASS_TO_IDX["person"] in lbls:
            preferred.append(i)
        elif len(lbls) > 0:
            fallback.append(i)
    pool = preferred + fallback
    if not pool:
        pool = list(range(len(ds)))
    # Lấy bước đều qua tập để có cả cảnh ngày và cảnh đêm
    step = max(1, len(pool) // max(1, count * 3))
    ordered = pool[start_offset::step] + pool
    seen, chosen = [], []
    for idx in ordered:
        if idx not in seen:
            seen.append(idx)
            chosen.append(idx)
        if len(chosen) >= count:
            break
    return chosen


VIZ_CACHE: List[Dict[str, Any]] = []

if DATA_ROOT.is_dir() and LOADED_TRANSLATORS:
    model_names = list(LOADED_TRANSLATORS.keys())
    viz_indices = select_diverse_visual_indices(val_eval_ds, CONFIG["num_visual_samples"], CONFIG["viz_start_index"])

    # Chạy inference một lần và lưu đệm để dùng chung cho Hình 1, Hình 2 (Zoom) và Hình 3 (FCOS Overlay)
    for ds_idx in viz_indices:
        item = val_eval_ds[ds_idx]
        rgb = item["rgb"].to(DEVICE)          # [3, H, W]
        real_ir = item["ir_1ch"].to(DEVICE)   # [1, H, W]
        gen_outputs: Dict[str, Tensor] = OrderedDict()
        for name, trans in LOADED_TRANSLATORS.items():
            gen_outputs[name] = trans.translate_1ch(rgb.unsqueeze(0))[0].cpu()  # [1, H, W]
        VIZ_CACHE.append({
            "stem": item["stem"],
            "rgb": rgb.cpu(),
            "real_ir": real_ir.cpu(),
            "target": item["target"],
            "gen_1ch": gen_outputs,
        })

    # --- HÌNH 1: ĐỐI CHIẾU TOÀN KHUNG SẠCH (KHÔNG VẼ BOX CHE MẮT) ---
    n_rows = len(VIZ_CACHE)
    n_cols = 2 + len(model_names)
    fig1, axes1 = plt.subplots(n_rows, n_cols, figsize=(4.6 * n_cols, 3.8 * n_rows), squeeze=False)
    for r, rec in enumerate(VIZ_CACHE):
        axes1[r, 0].imshow(rec["rgb"].permute(1, 2, 0).numpy().clip(0, 1))
        lbl_names = ", ".join(sorted({FLIR_CLASSES[int(l)] for l in rec["target"]["labels"].tolist()})) or "none"
        axes1[r, 0].set_title(f"RGB Gốc ({lbl_names})\n{rec['stem']}", fontsize=9)

        axes1[r, 1].imshow(rec["real_ir"][0].numpy(), cmap="gray", vmin=0, vmax=1)
        axes1[r, 1].set_title("Real IR (1-ch thật)", fontsize=9)

        for c_off, name in enumerate(model_names):
            img_1ch = rec["gen_1ch"][name][0].numpy()
            axes1[r, 2 + c_off].imshow(img_1ch, cmap="gray", vmin=0, vmax=1)
            axes1[r, 2 + c_off].set_title(f"{name} (1-ch)", fontsize=9)

        for c in range(n_cols):
            axes1[r, c].axis("off")

    fig1.suptitle("HÌNH 1: ĐÁNH GIÁ CHẤT LƯỢNG ẢNH IR 1 KÊNH SINH RA BẰNG MẮT THƯỜNG (TOÀN KHUNG)", fontsize=13, y=1.005)
    plt.tight_layout()
    out_fig1 = OUTPUT_DIR / "fig1_visual_fullframe_clean.png"
    fig1.savefig(out_fig1, dpi=150, bbox_inches="tight")
    print(f"[SAVED] Hình 1 (Toàn khung sạch) -> {out_fig1}")
    plt.show()

    # --- HÌNH 2: PHÓNG TO CẬN CẢNH VẬT THỂ NHỎ (PERSON / BICYCLE / CAR) ---
    zoom_candidates = []
    for rec in VIZ_CACHE:
        boxes = rec["target"]["boxes"]
        labels = rec["target"]["labels"]
        for b_i in range(boxes.shape[0]):
            x1, y1, x2, y2 = boxes[b_i].tolist()
            area = (x2 - x1) * (y2 - y1)
            cls_id = int(labels[b_i])
            if area >= 64.0:
                # Ưu tiên bicycle (0), person (1), car (2), rồi theo kích thước vừa/nhỏ
                prio = 0 if cls_id == FLIR_CLASS_TO_IDX["bicycle"] else (1 if cls_id == FLIR_CLASS_TO_IDX["person"] else 2)
                zoom_candidates.append((prio, area, rec, [x1, y1, x2, y2], FLIR_CLASSES[cls_id]))
    zoom_candidates.sort(key=lambda t: (t[0], abs(t[1] - 1600.0)))
    chosen_zooms = zoom_candidates[: CONFIG["num_zoom_objects"]]

    if chosen_zooms:
        fig2, axes2 = plt.subplots(len(chosen_zooms), n_cols, figsize=(3.6 * n_cols, 3.4 * len(chosen_zooms)), squeeze=False)
        for r, (_, area, rec, box, cls_name) in enumerate(chosen_zooms):
            _, H, W = rec["rgb"].shape
            x1, y1, x2, y2 = box
            cx, cy = 0.5 * (x1 + x2), 0.5 * (y1 + y2)
            half = max(44.0, max(x2 - x1, y2 - y1) * 1.45)
            cx1, cy1 = max(0, int(cx - half)), max(0, int(cy - half))
            cx2, cy2 = min(W, int(cx + half)), min(H, int(cy + half))

            axes2[r, 0].imshow(rec["rgb"][:, cy1:cy2, cx1:cx2].permute(1, 2, 0).numpy().clip(0, 1))
            axes2[r, 0].add_patch(patches.Rectangle((x1 - cx1, y1 - cy1), x2 - x1, y2 - y1, linewidth=1.3, edgecolor="#00ff66", facecolor="none"))
            axes2[r, 0].set_title(f"RGB Zoom: {cls_name}\n({int(x2-x1)}×{int(y2-y1)} px)", fontsize=9)

            axes2[r, 1].imshow(rec["real_ir"][0, cy1:cy2, cx1:cx2].numpy(), cmap="gray", vmin=0, vmax=1)
            axes2[r, 1].set_title("Real IR Zoom", fontsize=9)

            for c_off, name in enumerate(model_names):
                axes2[r, 2 + c_off].imshow(rec["gen_1ch"][name][0, cy1:cy2, cx1:cx2].numpy(), cmap="gray", vmin=0, vmax=1)
                axes2[r, 2 + c_off].set_title(f"{name} Zoom", fontsize=9)

            for c in range(n_cols):
                axes2[r, c].axis("off")

        fig2.suptitle("HÌNH 2: SOI CẬN CẢNH BIÊN & ĐỘ TƯƠNG PHẢN VẬT THỂ NHỎ (BICYCLE / PERSON / CAR)", fontsize=13, y=1.005)
        plt.tight_layout()
        out_fig2 = OUTPUT_DIR / "fig2_visual_small_object_zoom.png"
        fig2.savefig(out_fig2, dpi=150, bbox_inches="tight")
        print(f"[SAVED] Hình 2 (Zoom vật thể nhỏ) -> {out_fig2}")
        plt.show()

In [ ]:
# ==============================================================================
# CELL 6: HÌNH 3 — TRỰC QUAN HÓA DỰ ĐOÁN CỦA FROZEN BEST FCOS TRÊN TỪNG MIỀN ẢNH GIẢ
# ==============================================================================
def _draw_boxes(ax, boxes: Tensor, labels: Tensor, scores: Optional[Tensor] = None, conf_min: float = 0.4):
    palette = {0: "#00ff66", 1: "#ffcc00", 2: "#00e5ff"}  # person=xanh lá, car=vàng, bicycle=xanh ngọc
    for idx in range(boxes.shape[0]):
        if scores is not None and float(scores[idx]) < conf_min:
            continue
        x1, y1, x2, y2 = boxes[idx].tolist()
        cls_id = int(labels[idx])
        color = palette.get(cls_id, "red")
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=1.4, edgecolor=color, facecolor="none"))
        cls_name = FLIR_CLASSES[cls_id] if 0 <= cls_id < len(FLIR_CLASSES) else str(cls_id)
        caption = f"{cls_name}" if scores is None else f"{cls_name}:{float(scores[idx]):.2f}"
        ax.text(x1, max(10, y1 - 3), caption, color="black", fontsize=7, bbox=dict(facecolor=color, alpha=0.82, pad=1, edgecolor="none"))


if VIZ_CACHE and FROZEN_FCOS_JUDGE is not None:
    model_names = list(LOADED_TRANSLATORS.keys())
    n_rows = min(len(VIZ_CACHE), 6)
    n_cols = 2 + len(model_names)
    conf_thr = float(CONFIG["viz_fcos_conf_thresh"])

    fig3, axes3 = plt.subplots(n_rows, n_cols, figsize=(4.6 * n_cols, 3.8 * n_rows), squeeze=False)
    for r in range(n_rows):
        rec = VIZ_CACHE[r]
        tgt = rec["target"]

        axes3[r, 0].imshow(rec["rgb"].permute(1, 2, 0).numpy().clip(0, 1))
        _draw_boxes(axes3[r, 0], tgt["boxes"], tgt["labels"])
        axes3[r, 0].set_title(f"RGB + GT ({len(tgt['boxes'])} boxes)\n{rec['stem']}", fontsize=9)

        real_ir_3 = rec["real_ir"].repeat(3, 1, 1).to(DEVICE)
        p_real = FROZEN_FCOS_JUDGE.predict([real_ir_3])[0]
        n_real_det = int((p_real["scores"] >= conf_thr).sum().item())
        axes3[r, 1].imshow(rec["real_ir"][0].numpy(), cmap="gray", vmin=0, vmax=1)
        _draw_boxes(axes3[r, 1], p_real["boxes"].cpu(), p_real["labels"].cpu(), p_real["scores"].cpu(), conf_min=conf_thr)
        axes3[r, 1].set_title(f"Real IR + FCOS ({n_real_det} dets)", fontsize=9)

        for c_off, name in enumerate(model_names):
            g_1ch = rec["gen_1ch"][name]
            p_gen = FROZEN_FCOS_JUDGE.predict([g_1ch.repeat(3, 1, 1).to(DEVICE)])[0]
            n_det = int((p_gen["scores"] >= conf_thr).sum().item())
            axes3[r, 2 + c_off].imshow(g_1ch[0].numpy(), cmap="gray", vmin=0, vmax=1)
            _draw_boxes(axes3[r, 2 + c_off], p_gen["boxes"].cpu(), p_gen["labels"].cpu(), p_gen["scores"].cpu(), conf_min=conf_thr)
            axes3[r, 2 + c_off].set_title(f"{name} + FCOS ({n_det} dets)", fontsize=9)

        for c in range(n_cols):
            axes3[r, c].axis("off")

    fig3.suptitle(f"HÌNH 3: HỘP DỰ ĐOÁN CỦA FROZEN BEST FCOS (conf ≥ {conf_thr:.2f}) TRÊN CÁC MIỀN IR GIẢ", fontsize=13, y=1.005)
    plt.tight_layout()
    out_fig3 = OUTPUT_DIR / "fig3_fcos_predictions_overlay.png"
    fig3.savefig(out_fig3, dpi=150, bbox_inches="tight")
    print(f"[SAVED] Hình 3 (FCOS Overlay) -> {out_fig3}")
    plt.show()

In [ ]:
# ==============================================================================
# CELL 7: ĐO TOÀN BỘ METRICS ĐỊNH LƯỢNG (CÁCH 1 + CÁCH 2) TRÊN TOÀN BỘ ALIGN_VALIDATION.TXT
# ==============================================================================
@torch.no_grad()
def run_full_multi_model_benchmark(cfg: Dict[str, Any]) -> Dict[str, Any]:
    eval_ds = val_eval_ds
    if cfg.get("eval_max_images") is not None and int(cfg["eval_max_images"]) < len(val_eval_ds):
        eval_ds = Subset(val_eval_ds, list(range(int(cfg["eval_max_images"]))))

    val_loader = DataLoader(
        eval_ds,
        batch_size=cfg["eval_batch_size"],
        shuffle=False,
        num_workers=cfg["num_workers"],
        collate_fn=val_collate_fn,
    )
    inc_extractor = InceptionPool3Extractor().to(DEVICE)
    model_names = list(LOADED_TRANSLATORS.keys())
    print(f"[BENCHMARK] Bắt đầu đo đầy đủ 2 hệ metrics trên {len(eval_ds)} ảnh validation cho: {model_names}")

    # 1) Trích xuất đặc trưng Real-IR 1 lần duy nhất (tiết kiệm 3x thời gian InceptionV3!)
    t_real = time.time()
    real_inc_list: List[np.ndarray] = []
    real_hf_energies: List[float] = []
    real_contrasts_all: List[float] = []
    real_fcos_preds: List[Dict[str, Tensor]] = []
    all_targets_cpu: List[Dict[str, Tensor]] = []

    for batch in val_loader:
        fcos_real_batch: List[Tensor] = []
        for real_ir_cpu, tgt in zip(batch["ir_1ch"], batch["targets"]):
            real_ir = real_ir_cpu.unsqueeze(0).to(DEVICE)
            real_inc_list.append(inc_extractor(real_ir).cpu().numpy())
            real_hf_energies.append(_laplacian_energy(real_ir))
            for _, c_r in _signed_box_ring_contrasts(real_ir[0, 0], tgt["boxes"], tgt["labels"]):
                real_contrasts_all.append(c_r)
            fcos_real_batch.append(real_ir.squeeze(0).repeat(3, 1, 1))
            all_targets_cpu.append({k: v.detach().cpu() for k, v in tgt.items() if isinstance(v, Tensor)})

        if cfg.get("eval_real_ir_upper_bound") and FROZEN_FCOS_JUDGE is not None:
            preds = FROZEN_FCOS_JUDGE.predict(fcos_real_batch)
            real_fcos_preds.extend({k: v.detach().cpu() for k, v in p.items()} for p in preds)

    real_inc_feats = np.concatenate(real_inc_list, axis=0)
    mean_real_hf = float(np.mean(real_hf_energies))

    report: Dict[str, Any] = {
        "protocol": {
            "split": "align_validation.txt",
            "num_eval_images": len(eval_ds),
            "baseline_name": cfg.get("baseline_name", "CycleGAN"),
            "fcos_checkpoint": str(cfg.get("best_fcos_checkpoint")),
            "fcos_format": None if FROZEN_FCOS_JUDGE is None else FROZEN_FCOS_JUDGE.fmt_name,
        },
        "real_ir_upper_bound": None,
        "models": OrderedDict(),
    }

    if real_fcos_preds:
        real_pl = {
            f"conf_{c:.2f}": compute_pseudo_label_quality(real_fcos_preds, all_targets_cpu, conf_thresh=c, iou_thresh=0.5)
            for c in cfg["pseudo_label_confs"]
        }
        report["real_ir_upper_bound"] = {
            "method": "Real IR (Validation Anchor)",
            "elapsed_sec": round(time.time() - t_real, 2),
            "fcos_detection_metrics": compute_coco_detection_metrics(real_fcos_preds, all_targets_cpu),
            "pseudo_label_metrics": real_pl,
        }
        print(f"  ✓ Xong mốc trần Real IR ({report['real_ir_upper_bound']['elapsed_sec']}s) — AP50 = {report['real_ir_upper_bound']['fcos_detection_metrics']['AP50']:.4f}")

    # 2) Đánh giá từng Generator trên cùng tập validation
    for name, translator in LOADED_TRANSLATORS.items():
        t_m = time.time()
        fake_inc_list: List[np.ndarray] = []
        edge_cos_scores, ssim_scores, psnr_scores = [], [], []
        fake_hf_energies: List[float] = []
        fake_contrasts_all: List[float] = []
        fake_person_contrasts: List[float] = []
        fcos_preds: List[Dict[str, Tensor]] = []

        for batch in val_loader:
            fcos_fake_batch: List[Tensor] = []
            for rgb_cpu, real_ir_cpu, tgt in zip(batch["rgb"], batch["ir_1ch"], batch["targets"]):
                rgb = rgb_cpu.unsqueeze(0).to(DEVICE)
                real_ir = real_ir_cpu.unsqueeze(0).to(DEVICE)
                fake_ir = translator.translate_1ch(rgb)  # [1, 1, H, W] (ĐÚNG 1 KÊNH)
                assert fake_ir.shape[1] == 1

                fake_inc_list.append(inc_extractor(fake_ir).cpu().numpy())

                rgb_luma = 0.2989 * rgb[:, 0:1] + 0.5870 * rgb[:, 1:2] + 0.1140 * rgb[:, 2:3]
                g_rgb = _sobel_magnitude(rgb_luma).flatten(1)
                g_fake = _sobel_magnitude(fake_ir).flatten(1)
                edge_cos_scores.append(float(F.cosine_similarity(g_rgb, g_fake, dim=1).item()))

                fake_hf_energies.append(_laplacian_energy(fake_ir))
                mse = float(F.mse_loss(fake_ir, real_ir).item())
                psnr_scores.append(10.0 * math.log10(1.0 / max(mse, 1e-10)))
                ssim_scores.append(_ssim_1ch(fake_ir, real_ir))

                for lbl_f, c_f in _signed_box_ring_contrasts(fake_ir[0, 0], tgt["boxes"], tgt["labels"]):
                    fake_contrasts_all.append(c_f)
                    if lbl_f == FLIR_CLASS_TO_IDX["person"]:
                        fake_person_contrasts.append(c_f)

                fcos_fake_batch.append(fake_ir.squeeze(0).repeat(3, 1, 1))

            if FROZEN_FCOS_JUDGE is not None:
                preds = FROZEN_FCOS_JUDGE.predict(fcos_fake_batch)
                fcos_preds.extend({k: v.detach().cpu() for k, v in p.items()} for p in preds)

        fake_inc_feats = np.concatenate(fake_inc_list, axis=0)
        fid_kid = compute_fid_and_kid(
            real_inc_feats, fake_inc_feats,
            kid_subsets=cfg["kid_subsets"], kid_subset_size=cfg["kid_subset_size"], seed=cfg["seed"],
        )
        gen_metrics = {
            **fid_kid,
            "Edge_Structure_CosSim": float(np.mean(edge_cos_scores)),
            "Thermal_Contrast_W1": _wasserstein_1d(real_contrasts_all, fake_contrasts_all),
            "Person_Positive_Contrast_Ratio": (
                float(np.mean([c > 0 for c in fake_person_contrasts])) if fake_person_contrasts else float("nan")
            ),
            "HighFreq_Energy_Ratio_Fake_over_Real": float(np.mean(fake_hf_energies)) / max(mean_real_hf, 1e-12),
            "Ref_Aligned_SSIM": float(np.mean(ssim_scores)),
            "Ref_Aligned_PSNR_dB": float(np.mean(psnr_scores)),
        }

        det_metrics = compute_coco_detection_metrics(fcos_preds, all_targets_cpu) if fcos_preds else None
        pl_metrics = (
            {
                f"conf_{c:.2f}": compute_pseudo_label_quality(fcos_preds, all_targets_cpu, conf_thresh=c, iou_thresh=0.5)
                for c in cfg["pseudo_label_confs"]
            }
            if fcos_preds else {}
        )
        elapsed = round(time.time() - t_m, 2)
        report["models"][name] = {
            "method": name,
            "checkpoint_info": translator.info_tag,
            "elapsed_sec": elapsed,
            "generation_metrics": gen_metrics,
            "fcos_detection_metrics": det_metrics,
            "pseudo_label_metrics": pl_metrics,
        }
        ap_str = f" | FCOS AP50 = {det_metrics['AP50']:.4f}" if det_metrics else ""
        print(f"  ✓ Xong {name:14s} ({elapsed:5.1f}s) | FID = {gen_metrics['FID']:.2f} | KID×1e3 = {gen_metrics['KID_x1000_mean']:.2f}{ap_str}")

    out_json = OUTPUT_DIR / "full_comparison_report.json"
    out_json.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"\n[SAVED] Báo cáo đầy đủ JSON -> {out_json}")
    return report


BENCHMARK_REPORT: Optional[Dict[str, Any]] = None
if DATA_ROOT.is_dir() and LOADED_TRANSLATORS:
    BENCHMARK_REPORT = run_full_multi_model_benchmark(CONFIG)

In [ ]:
# ==============================================================================
# CELL 8: IN BẢNG ĐỐI CHIẾU TỔNG HỢP, XUẤT CSV & VẼ BIỂU ĐỒ SO SÁNH CÁC MÔ HÌNH
# ==============================================================================
def render_tables_csv_and_charts(report: Dict[str, Any]) -> None:
    models = report["models"]
    names = list(models.keys())
    base_name = report["protocol"].get("baseline_name", "CycleGAN")
    real_anchor = report.get("real_ir_upper_bound")
    csv_rows: List[Dict[str, Any]] = []

    def _pick_winner(vals: Dict[str, float], mode: str) -> Optional[str]:
        valid = {k: v for k, v in vals.items() if np.isfinite(v)}
        if not valid:
            return None
        if mode == "higher":
            return max(valid, key=valid.get)
        if mode == "lower":
            return min(valid, key=valid.get)
        if mode == "near_one":
            return min(valid, key=lambda k: abs(valid[k] - 1.0))
        return None

    # --- BẢNG 1: IMAGE GENERATION METRICS ---
    gen_specs = [
        ("FID (↓ càng thấp càng thật)", "FID", "lower"),
        ("KID × 10^3 (↓ không chệch cỡ mẫu nhỏ)", "KID_x1000_mean", "lower"),
        ("Edge Structure CosSim (↑ giữ biên vật thể)", "Edge_Structure_CosSim", "higher"),
        ("Thermal Contrast W1 (↓ khớp tương phản nhiệt)", "Thermal_Contrast_W1", "lower"),
        ("Person Positive Contrast (↑ người sáng hơn nền)", "Person_Positive_Contrast_Ratio", "higher"),
        ("High-Freq Ratio Fake/Real (≈1.0 ít giấu tin)", "HighFreq_Energy_Ratio_Fake_over_Real", "near_one"),
        ("Ref Val SSIM (↑ tham chiếu)", "Ref_Aligned_SSIM", "higher"),
        ("Ref Val PSNR dB (↑ tham chiếu)", "Ref_Aligned_PSNR_dB", "higher"),
    ]
    col_w = 15
    line_w = 48 + (col_w + 3) * len(names) + 18
    print("\n" + "=" * line_w)
    print("BẢNG 1 — CÁCH EVAL 1: METRICS SINH ẢNH HỒNG NGOẠI 1 KÊNH (IMAGE GENERATION METRICS)")
    print("=" * line_w)
    header = f"{'Metric':46s} | " + " | ".join(f"{n:>{col_w}s}" for n in names) + " | Tốt nhất"
    print(header)
    print("-" * line_w)
    for label, key, mode in gen_specs:
        vals = {n: float((models[n].get("generation_metrics") or {}).get(key, float("nan"))) for n in names}
        win = _pick_winner(vals, mode)
        row_str = f"{label:46s} | " + " | ".join(f"{vals[n]:{col_w}.4f}" for n in names) + f" | ★ {win or '-'}"
        print(row_str)
        csv_rows.append({"Group": "Generation", "Metric": key, "Real_IR": "", **{n: vals[n] for n in names}, "Winner": win or ""})
    print("=" * line_w)

    # --- BẢNG 2: FROZEN BEST FCOS DETECTOR & PSEUDO-LABEL QUALITY ---
    has_det = any(models[n].get("fcos_detection_metrics") for n in names)
    if has_det:
        det_specs = [
            ("COCO mAP @[0.50:0.95]", "AP"),
            ("COCO mAP @0.50 (AP50)", "AP50"),
            ("COCO mAP @0.75 (AP75)", "AP75"),
            ("COCO AP_small", "AP_small"),
            ("COCO AR@100", "AR100"),
            ("AP50 / person", "AP50/person"),
            ("AP50 / car", "AP50/car"),
            ("AP50 / bicycle (vật mảnh dễ mất)", "AP50/bicycle"),
        ]
        line_w2 = 42 + 17 + (col_w + 3) * len(names) + 18
        print("\n" + "=" * line_w2)
        print("BẢNG 2 — CÁCH EVAL 2: ĐÓNG BĂNG BEST FCOS DETECTOR CHẤM DETECTION & CHẤT LƯỢNG NHÃN GIẢ")
        print("=" * line_w2)
        header2 = f"{'Metric (↑ càng cao càng tốt)':40s} | {'Real IR (Trần)':>14s} | " + " | ".join(f"{n:>{col_w}s}" for n in names) + " | Tốt nhất"
        print(header2)
        print("-" * line_w2)

        d_real = (real_anchor or {}).get("fcos_detection_metrics") or {}
        for label, key in det_specs:
            vr = float(d_real.get(key, float("nan")))
            vals = {n: float((models[n].get("fcos_detection_metrics") or {}).get(key, float("nan"))) for n in names}
            win = _pick_winner(vals, "higher")
            print(f"{label:40s} | {vr:14.4f} | " + " | ".join(f"{vals[n]:{col_w}.4f}" for n in names) + f" | ★ {win or '-'}")
            csv_rows.append({"Group": "FCOS_Det", "Metric": key, "Real_IR": vr, **{n: vals[n] for n in names}, "Winner": win or ""})

        conf_keys = sorted(next((models[n]["pseudo_label_metrics"].keys() for n in names if models[n].get("pseudo_label_metrics")), []))
        for ck in conf_keys:
            print("-" * line_w2)
            p_real = ((real_anchor or {}).get("pseudo_label_metrics") or {}).get(ck, {})
            for pl_lbl, pl_k in (
                (f"Nhãn giả Precision ({ck})", "PL_Precision"),
                (f"Nhãn giả Recall    ({ck})", "PL_Recall"),
                (f"Nhãn giả F1-Score  ({ck})", "PL_F1"),
                (f"Nhãn giả Mean IoU  ({ck})", "PL_MeanMatchedIoU"),
            ):
                vr = float(p_real.get(pl_k, float("nan")))
                vals = {n: float((models[n].get("pseudo_label_metrics") or {}).get(ck, {}).get(pl_k, float("nan"))) for n in names}
                win = _pick_winner(vals, "higher")
                print(f"{pl_lbl:40s} | {vr:14.4f} | " + " | ".join(f"{vals[n]:{col_w}.4f}" for n in names) + f" | ★ {win or '-'}")
                csv_rows.append({"Group": f"PL_{ck}", "Metric": pl_k, "Real_IR": vr, **{n: vals[n] for n in names}, "Winner": win or ""})
        print("=" * line_w2)

        # In tóm tắt chênh lệch Δ so với baseline (mặc định CycleGAN)
        if base_name in models:
            print(f"\n[TÓM TẮT ĐỘ CHÊNH LỆCH (Δ) SO VỚI MỐC '{base_name}']:")
            base_g = models[base_name]["generation_metrics"]
            base_d = models[base_name]["fcos_detection_metrics"]
            base_pl50 = (models[base_name]["pseudo_label_metrics"] or {}).get("conf_0.50", {})
            for n in names:
                if n == base_name:
                    continue
                g = models[n]["generation_metrics"]
                d = models[n]["fcos_detection_metrics"]
                pl50 = (models[n]["pseudo_label_metrics"] or {}).get("conf_0.50", {})
                print(
                    f"  • {n:14s}: "
                    f"ΔFID={g['FID']-base_g['FID']:+.2f} | "
                    f"ΔKID×1e3={g['KID_x1000_mean']-base_g['KID_x1000_mean']:+.2f} | "
                    f"ΔAP50={d['AP50']-base_d['AP50']:+.4f} | "
                    f"ΔmAP={d['AP']-base_d['AP']:+.4f} | "
                    f"ΔAP50(bicycle)={d['AP50/bicycle']-base_d['AP50/bicycle']:+.4f} | "
                    f"ΔPL_F1@0.5={pl50.get('PL_F1', float('nan'))-base_pl50.get('PL_F1', float('nan')):+.4f}"
                )

    # Xuất CSV tổng hợp
    csv_path = OUTPUT_DIR / "summary_table.csv"
    fieldnames = ["Group", "Metric", "Real_IR", *names, "Winner"]
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(csv_rows)
    print(f"\n[SAVED] Bảng CSV tổng hợp -> {csv_path}")

    # --- HÌNH 4: BIỂU ĐỒ CỘT TỔNG HỢP SO SÁNH CÁC MÔ HÌNH ---
    fig4, axes4 = plt.subplots(1, 3, figsize=(18, 4.8))
    x = np.arange(len(names))
    colors = ["#4c78a8", "#f58518", "#54a24b", "#e45756", "#72b7b2"][: len(names)]

    # Panel 1: FID (↓) & KID×10^3 (↓)
    fids = [models[n]["generation_metrics"]["FID"] for n in names]
    bars0 = axes4[0].bar(x, fids, color=colors, width=0.55)
    axes4[0].set_xticks(x)
    axes4[0].set_xticklabels(names, rotation=15)
    axes4[0].set_title("FID ↓ (Càng thấp càng giống IR thật)", fontsize=11)
    axes4[0].grid(axis="y", linestyle="--", alpha=0.4)
    for b in bars0:
        axes4[0].text(b.get_x() + b.get_width() / 2, b.get_height(), f"{b.get_height():.1f}", ha="center", va="bottom", fontsize=9)

    if has_det:
        # Panel 2: Frozen FCOS mAP & AP50 (↑) + đường trần Real IR
        w = 0.35
        ap50s = [models[n]["fcos_detection_metrics"]["AP50"] for n in names]
        maps = [models[n]["fcos_detection_metrics"]["AP"] for n in names]
        b1 = axes4[1].bar(x - w / 2, ap50s, width=w, label="AP50", color="#2a9d8f")
        b2 = axes4[1].bar(x + w / 2, maps, width=w, label="mAP@[.50:.95]", color="#264653")
        if real_anchor and real_anchor.get("fcos_detection_metrics"):
            axes4[1].axhline(real_anchor["fcos_detection_metrics"]["AP50"], color="red", linestyle="--", linewidth=1.3, label="Real IR AP50 (Trần)")
        axes4[1].set_xticks(x)
        axes4[1].set_xticklabels(names, rotation=15)
        axes4[1].set_title("Frozen Best FCOS Detection ↑", fontsize=11)
        axes4[1].legend(fontsize=8)
        axes4[1].grid(axis="y", linestyle="--", alpha=0.4)
        for b in list(b1) + list(b2):
            axes4[1].text(b.get_x() + b.get_width() / 2, b.get_height(), f"{b.get_height():.3f}", ha="center", va="bottom", fontsize=8)

        # Panel 3: Per-class AP50 (person, car, bicycle)
        w3 = 0.25
        for c_i, (cls_name, col_c) in enumerate((("person", "#2a9d8f"), ("car", "#e9c46a"), ("bicycle", "#e76f51"))):
            vals_c = [models[n]["fcos_detection_metrics"].get(f"AP50/{cls_name}", 0.0) for n in names]
            bc = axes4[2].bar(x + (c_i - 1) * w3, vals_c, width=w3, label=cls_name, color=col_c)
            for b in bc:
                axes4[2].text(b.get_x() + b.get_width() / 2, b.get_height(), f"{b.get_height():.2f}", ha="center", va="bottom", fontsize=7.5)
        axes4[2].set_xticks(x)
        axes4[2].set_xticklabels(names, rotation=15)
        axes4[2].set_title("Frozen FCOS AP50 theo từng lớp ↑", fontsize=11)
        axes4[2].legend(fontsize=8)
        axes4[2].grid(axis="y", linestyle="--", alpha=0.4)
    else:
        axes4[1].axis("off")
        axes4[2].axis("off")

    plt.tight_layout()
    out_fig4 = OUTPUT_DIR / "fig4_metrics_summary_chart.png"
    fig4.savefig(out_fig4, dpi=150, bbox_inches="tight")
    print(f"[SAVED] Hình 4 (Biểu đồ tổng hợp metrics) -> {out_fig4}")
    plt.show()


if BENCHMARK_REPORT is not None:
    render_tables_csv_and_charts(BENCHMARK_REPORT)